# MQTT IoT adversarial ML reproduction study

This notebook reproduces the software workflow from `HenryCWong/adversarialBERTMessages`, the MIT-licensed code for the paper *Man-in-the-Middle Attacks on MQTT-based IoT Using BERT Based Adversarial Message Generation*.

The workflow generates MQTT-like adversarial messages from sample telemetry data, runs several anomaly-detection classifiers, and compares accuracy and false negatives as the perturbation setting changes. This notebook is only for local controlled study.


## Dependencies

Install dependencies from the repository root with `pip install -r requirements.txt` before running this notebook.


In [ ]:
import nltk
nltk.download("popular")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import torch
import transformers as ppb
import warnings
from nltk.corpus import wordnet as wn
wordnet = wn
import random
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# For DistilBERT:
model_class, tokenizer_class, pretrained_weights = (ppb.DistilBertModel, ppb.DistilBertTokenizer, 'distilbert-base-uncased')
# Load pretrained model/tokenizer
tokenizer = tokenizer_class.from_pretrained(pretrained_weights)
model = model_class.from_pretrained(pretrained_weights)

## Function Definitions

In [ ]:
# Generate modified values for non-numeric fields.
# X is a pandas Series.
# gamma controls the perturbation setting.
# Returns a series of modified values.
def generatorWords(X,gamma):

  fakeX = []

  if len(X.unique()) < len(X)/2: #if the number of unique values is less than half of the set
    uniqueVals = X.unique()
    fakeX = []
    for i in range(len(X)):
      fakeX.append(random.choice(uniqueVals))
  else:
    for index, value in X.items():
      words = value.split(" ")
      adjs = []
      chosen_word = ""
      pos_all = dict()
      antonyms = []

      # Prefer adjective substitutions where possible.
      for w in words:
        pos_l = set()
        for tmp in wn.synsets(w):
            if tmp.name().split('.')[0] == w:
                pos_l.add(tmp.pos())
        pos_all[w] = pos_l

      # Identify adjective candidates.
      for a in pos_all:
        if "a" in pos_all[a]:
          adjs.append(a)

      # Choose a word to alter.
      if adjs:
        chosen_word = adjs[0] if len(adjs) > 1 else adjs[0]
      else:
        chosen_word = words[0] if len(words) > 1 else words[0]

      # Find antonyms for the selected word.
      for syn in wordnet.synsets(chosen_word):
        for lm in syn.lemmas():
            if lm.antonyms():
                antonyms.append(lm.antonyms()[0].name())
      if antonyms:
        new = [antonyms[gamma] if x==chosen_word else x for x in words]
      else:
        new = [x if x==chosen_word else x for x in words]
      fakeX.append(" ".join(new))
  return pd.Series(fakeX)


In [ ]:
# Generate modified values for numeric fields.
# X is a numeric series.
def generatorNum(X,gamma):
  xHat = np.mean(X)
  sx = np.std(X)
  randomUpper = (xHat) + (sx * gamma)
  randomLower = (xHat) - (sx * gamma)
  beta = []
  for i in range(int(len(X))):
      beta.append(random.randrange(int(randomLower),int(randomUpper)))
  beta = list(map(str, beta))
  return pd.Series(beta)

In [ ]:
# Route each column through the appropriate generator.
# Return generated and original records with binary labels.
# Label 0 is original data; label 1 is generated data.
def generator(df,gamma):
  mDF = df.copy()
  mDF["temperature"] = generatorNum(df["temperature"],gamma)
  mDF["outlook"] = generatorWords(df["outlook"],0)
  mDF["wind"] = generatorWords(df["wind"],0)
  mDF["humidity"] = generatorNum(df["humidity"],gamma)
  fakeData = addLabel(makeSentence(mDF),1) #column 0 is sentences, column 1 is label
  realData = addLabel(makeSentence(df),0)  #Label 0 means data, label 1 means fake data
  return(pd.concat([fakeData,realData],axis=0),fakeData)

In [ ]:
# Merge dataframe columns into a single sentence-like representation.
# Return one text field per row.
def makeSentence(df):
  # df is a dataframe.
  df = df.copy().applymap(str)
  seriesA = df[df.columns[0:]].apply(lambda x: ','.join(x.dropna().astype(str)),axis=1)
  return seriesA

In [ ]:
# Add a label column for classification.
# Return the text series and labels as a dataframe.
def addLabel(series,label):
  # Takes a pandas Series and label value.
  # Returns a dataframe.
  labels = []
  [labels.append(label) for i in range(len(series))]
  labels = pd.Series(labels)
  return pd.concat([series,labels],axis=1)

In [ ]:
# Discriminator step for evaluating generated messages.
# Return the updated gamma and evaluation values.
def discriminator(X, gamma, rho):
  # Takes a dataframe of sentence-like records and labels.
  # Returns the next gamma value.
  score,indexes,fn,total = BERT(X)
  print("score: " + str(score))
  return (score,gamma + rho,indexes,fn,total)

In [ ]:
def BERT(df):
  # df contains sentence-like records and labels.
  # BERT feature extraction returns accuracy and false-negative indexes.

  # Tokenize records.
  tokenized = df[0].apply((lambda x: tokenizer.encode(x, add_special_tokens=True)))
  max_len = 0
  for i in tokenized.values:
      if len(i) > max_len:
          max_len = len(i)

  padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values]) #pad
  attention_mask = np.where(padded != 0, 1, 0)                            #apply masks
  input_ids = torch.tensor(padded)
  attention_mask = torch.tensor(attention_mask)                           #put masks through torch tensor

  # Run records through BERT.
  with torch.no_grad():
      last_hidden_states = model(input_ids, attention_mask=attention_mask)
  features = last_hidden_states[0][:,0,:].numpy()
  labels = df[1]
  return(regression(features,labels))

In [ ]:
# Evaluate BERT features with a multilayer perceptron.
# Return accuracy and false-negative indexes.
def regression(features,labels):
  train_features, test_features, train_labels, test_labels = train_test_split(features, labels, random_state=42)
  NeuralNetwork = MLPClassifier(solver='adam', activation="logistic", random_state=1,max_iter=5000)
  NeuralNetwork.fit(train_features, train_labels)
  y2 = NeuralNetwork.predict(test_features)
  y2 = pd.Series(NeuralNetwork.predict(test_features))
  i=0
  indexes = []
  for index, value in test_labels.items():
    if value == 1:
      if y2[i] == 0:
        indexes.append(index)
    i+=1
  tn, fp, fn, tp = confusion_matrix(test_labels, y2).ravel()
  total = len(test_labels)
  return (NeuralNetwork.score(test_features,test_labels),indexes,fn,total)

In [ ]:
def indexToEntry(X,indexes):
  # Return dataframe rows for the selected indexes.
  entries = []
  for i in indexes:
    entries.append(X.iloc[i])
  return(pd.DataFrame(entries))

In [ ]:
def logisticRegression(train_features, test_features, train_labels, test_labels):
  lr_clf = LogisticRegression(max_iter=500)
  lr_clf.fit(train_features, train_labels)
  y_pred = lr_clf.predict(test_features)
  score = lr_clf.score(test_features, test_labels)
  tn, fp, fn, tp = confusion_matrix(test_labels, y_pred).ravel()
  return(fn,score)

In [ ]:
def RandomForest(train_features, test_features, train_labels, test_labels):
  RandForest = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0)
  RandForest.fit(train_features, train_labels)
  forest_pred = RandForest.predict(test_features)
  score = RandForest.score(test_features,test_labels)
  tn, fp, fn, tp = confusion_matrix(test_labels, forest_pred).ravel()
  return(fn,score)


In [ ]:
def KNN(train_features, test_features, train_labels, test_labels):
  # K Nearest Neighbor with k=1.
  knn = KNeighborsClassifier(n_neighbors = 1)
  knn.fit(train_features,train_labels)
  score = knn.score(test_features, test_labels)
  y_pred = knn.predict(test_features)
  tn, fp, fn, tp = confusion_matrix(test_labels, y_pred).ravel()
  return(fn,score)

In [ ]:
def MLP(train_features, test_features, train_labels, test_labels):
  NeuralNetwork = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=1)
  NeuralNetwork.fit(train_features, train_labels)
  nn_pred = NeuralNetwork.predict(test_features)
  score = NeuralNetwork.score(test_features,test_labels)
  tn, fp, fn, tp = confusion_matrix(test_labels, nn_pred).ravel()
  return(fn,score)

In [ ]:
def SVM(train_features, test_features, train_labels, test_labels):
  clf = make_pipeline(StandardScaler(), SVC(gamma='auto'))
  clf.fit(train_features,train_labels)
  svmPred = clf.predict(test_features)
  score = clf.score(test_features,test_labels)
  tn, fp, fn, tp = confusion_matrix(test_labels, svmPred).ravel()
  return(fn,score)

In [ ]:
def tests(df,mDF):
  ogData = addLabel(makeSentence(df),0)
  testData = pd.concat([ogData,mDF])
  tokenized = testData[0].apply((lambda x: tokenizer.encode(x, add_special_tokens=True)))
  max_len = 0
  for i in tokenized.values:
      if len(i) > max_len:
          max_len = len(i)

  padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])

  train_features, test_features, train_labels, test_labels = train_test_split(padded, testData[1], random_state=42)
  lrFN,lrScore = logisticRegression(train_features, test_features, train_labels, test_labels)
  rfFN,rfScore = RandomForest(train_features, test_features, train_labels, test_labels)
  knnFN,knnScore = KNN(train_features, test_features, train_labels, test_labels)
  mlpFN,mlpScore = MLP(train_features, test_features, train_labels, test_labels)
  svmFN,svmScore = SVM(train_features, test_features, train_labels, test_labels)
  return(lrFN,lrScore,rfFN,rfScore,knnFN,knnScore,mlpFN,mlpScore,svmFN,svmScore)

## Data Processing

In [ ]:
from pathlib import Path

repo_data = Path('../data/weather_sample_data.csv')
df = pd.read_csv(repo_data if repo_data.exists() else 'https://raw.githubusercontent.com/HenryCWong/adversarialBERTMessages/master/weather_sample_data.csv', delimiter=',')


In [ ]:
df.drop("Unnamed: 0",axis=1,inplace=True)

In [ ]:
df.head()

## Algorithm

In [ ]:
X,mDF = generator(df,1)

In [ ]:
mDF

In [ ]:
tests(df,mDF) #logistic regression, random forest, knn, mlp, svm

In [ ]:
score,gamma,indexes,jp,total = discriminator(X,1,1)

In [ ]:
print(jp)
print(total)

In [ ]:
malDF = indexToEntry(X,indexes)

In [ ]:
malDF.head()

### Main Evaluation Loop

In [ ]:

gamma = 1
rho = .1
# Collect metrics for plots and comparison.
sets = []
scores = []
fns = []
totals = []
gammas = []
lrFNs = []
rfFNs = []
knnFNs = []
mlpFNs = []
svmFNs = []
lrScores = []
rfScores = []
knnScores = []
mlpScores = []
svmScores = []
j = 0
while j < 30:
  X,mDF = generator(df,gamma)
  score,gamma, indexes,fn,total = discriminator(X,gamma,rho)
  scores.append((j,score))
  lrFN,lrScore,rfFN,rfScore,knnFN,knnScore,mlpFN,mlpScore,svmFN,svmScore = tests(df,mDF)
  # Store classifier metrics.
  lrFNs.append(lrFN)
  rfFNs.append(rfFN)
  knnFNs.append(knnFN)
  mlpFNs.append(mlpFN)
  svmFNs.append(svmFN)
  lrScores.append(lrScore)
  rfScores.append(rfScore)
  knnScores.append(knnScore)
  mlpScores.append(mlpScore)
  svmScores.append(svmScore)
  sets.append(indexToEntry(X,indexes))
  fns.append(fn)
  totals.append(total)
  gammas.append(gamma)
  j+=1


In [ ]:
sortedScores = sorted(scores, key=lambda tup: tup[1])

In [ ]:
index,score = sortedScores[0]
malDF = sets[index]
for i in range(1,7):
  index, score = sortedScores[i]
  malDF = pd.concat([malDF,sets[index]])

In [ ]:
malDF

In [ ]:
df["temperature"].std()

## Plots

In [ ]:
print(gammas)
print(fns)
print(totals)
print(scores)

In [ ]:
for a in scores:
  print(a[1])

In [ ]:
numScores = []
[numScores.append(a[1]) for a in scores]

In [ ]:
sns.set_style("whitegrid")
sns.palplot(sns.color_palette("cubehelix", 6))
sns.set_palette("cubehelix",6)

In [ ]:
a4_dims = (15, 7)
fig, ax = plt.subplots(figsize=a4_dims)
ax.set(xlabel='Gamma', ylabel='Accuracy')
sns.lineplot(ax=ax,x=gammas,y=numScores)


In [ ]:
a4_dims = (15, 7)
fig, ax = plt.subplots(figsize=a4_dims)
ax.set_xlabel('Gamma',fontsize=20);
ax.set_ylabel('Number of False Negatives',fontsize=20);
sns.lineplot(ax=ax,x=gammas,y=fns,markers=True)


In [ ]:
fpRatio = []
for i in range(len(fns)):
  fpRatio.append(fns[i]/totals[i])

In [ ]:
a4_dims = (15, 7)
fig, ax = plt.subplots(figsize=a4_dims)
ax.set(xlabel='Gamma', ylabel='False Negatives / Total Data Points')
sns.lineplot(ax=ax,x=gammas,y=fpRatio)


In [ ]:
d = {"gamma":gammas,"score":numScores,"fn":fns}
nDF = pd.DataFrame(data=d)

In [ ]:
a4_dims = (15, 7)

fig = plt.figure(figsize=a4_dims)
a1 = fig.add_axes([0,0,1,1])
a1.plot(nDF["gamma"],nDF["score"],color="orchid",marker="x",linewidth=10.0)
a1.set_ylabel('Accuracy',fontsize = 30)
a1.set_xlabel("Gamma",fontsize = 30)
a2 = a1.twinx()
a2.plot(nDF["gamma"],nDF["fn"],marker="s",linewidth=10.0)
a2.set_ylabel('False Negatives',fontsize = 30)
a2.set_xlabel("Gamma",fontsize = 30)
fig.legend(labels = ('Accuracy','False Negatives'),loc='upper center',prop={'size': 30})
plt.show()

In [ ]:
# Interpolation attempt.
from scipy.interpolate import interp1d


In [ ]:
f1 = interp1d(nDF["gamma"], nDF['score'],kind='cubic')
f2 = interp1d(nDF["gamma"], nDF['fn'],kind='cubic')

xnew = np.arange(1.1,4,.3)
df2 = pd.DataFrame()
df2['Weight_A'] = f1(xnew)
print(df2)
ax2 = df2.plot.line()
ax2.set_title('After interpolation')
ax2.set_xlabel("year")
ax2.set_ylabel("weight")


plt.show()

In [ ]:
d = {"Logistic Regression":lrFNs,"Random Forest":rfFNs,"K Nearest Neighbor":knnFNs,"Multi-Layer Perceptron":mlpFNs,"Support Vector Classification":svmFNs,"gamma":gammas}
mlstuffs = pd.DataFrame(d)

In [ ]:
print(svmScores)

In [ ]:
s = {"Logistic Regression":lrScores,"Random Forest":rfScores,"K Nearest Neighbor":knnScores,"Multi-Layer Perceptron":mlpScores,"Support Vector Classification":svmScores,"gamma":gammas}
scoreStuffs = pd.DataFrame(s)

In [ ]:
meltedScores = scoreStuffs.melt(id_vars=['gamma'])
meltedScores = scoreStuffs.melt(id_vars=['gamma'])
meltedScores.columns = ["gamma","Classification Model","Accuracy"]

In [ ]:
meltedML = mlstuffs.melt(id_vars=['gamma'])
meltedML.columns = ["gamma","Classification Model","Number of False Negatives"]

In [ ]:
meltedML.head()

In [ ]:
a4_dims = (15, 10)
fig, ax = plt.subplots(figsize=a4_dims)
ax.set_xlabel('Gamma',fontsize=20);
ax.set_ylabel('Number of False Negatives',fontsize=20);
sns.lineplot(ax=ax,x="gamma",y="Number of False Negatives",hue="Classification Model",data=meltedML)
# Plot false negatives across gamma values.

In [ ]:
a4_dims = (15, 10)
fig, ax = plt.subplots(figsize=a4_dims)
ax.set_xlabel('Gamma',fontsize=30);
ax.set_ylabel('Number of False Negatives',fontsize=30);
models = ["Logistic Regression","Random Forest","K Nearest Neighbor","Multi-Layer Perceptron","Support Vector Classification"]
markers = ["x","^","*","x","s"]
for i in range(len(models)):
  a = meltedML[meltedML["Classification Model"] == models[i]]
  if models[i] == "Multi-Layer Perceptron" or models[i] == "K Nearest Neighbor":
    plt.plot(a["gamma"],a["Number of False Negatives"],linewidth=10.0,marker=markers[i])
  else:
    plt.plot(a["gamma"],a["Number of False Negatives"],linewidth=1.0,marker=markers[i])
plt.legend(models,prop={'size': 20})




In [ ]:
meltedScores.head()

In [ ]:
a4_dims = (15, 7)
fig, ax = plt.subplots(figsize=a4_dims)
ax.set_xlabel('Gamma',fontsize=20);
ax.set_ylabel('Accuracy',fontsize=20);
sns.lineplot(ax=ax,x="gamma",y="Accuracy",hue="Classification Model",data=meltedScores)
# Plot false negatives across gamma values.

In [ ]:
a4_dims = (15, 10)
fig, ax = plt.subplots(figsize=a4_dims)
ax.set_xlabel('Gamma',fontsize=30);
ax.set_ylabel('Accuracy',fontsize=30);
models = ["Logistic Regression","Random Forest","K Nearest Neighbor","Multi-Layer Perceptron","Support Vector Classification"]
markers = ["x","^","*","x","s"]
for i in range(len(models)):
  a = meltedScores[meltedScores["Classification Model"] == models[i]]
  if models[i] == "Multi-Layer Perceptron" or models[i] == "Support Vector Classification":
    plt.plot(a["gamma"],a["Accuracy"],linewidth=10.0,marker=markers[i])
  else:
    plt.plot(a["gamma"],a["Accuracy"],linewidth=1.0,marker=markers[i])
plt.legend(models,prop={'size': 20})




## Classifier Checks

In [ ]:
ogData = addLabel(makeSentence(df),0)

In [ ]:
testData = pd.concat([ogData,malDF])

In [ ]:
testData

In [ ]:
testData[testData[1] == 1]

In [ ]:
tokenized = testData[0].apply((lambda x: tokenizer.encode(x, add_special_tokens=True)))

In [ ]:
max_len = 0
for i in tokenized.values:
    if len(i) > max_len:
        max_len = len(i)

padded = np.array([i + [0]*(max_len-len(i)) for i in tokenized.values])

In [ ]:
padded

In [ ]:
train_features, test_features, train_labels, test_labels = train_test_split(padded, testData[1], random_state=42)

In [ ]:
test_labels.value_counts()

### Logistic Regression Test

In [ ]:
lr_clf = LogisticRegression(max_iter=500)
lr_clf.fit(train_features, train_labels)
print(lr_clf.score(test_features, test_labels))
y_pred = lr_clf.predict(test_features)

In [ ]:
confusion_matrix(test_labels, y_pred)

### Random Forest Test

In [ ]:
RandForest = RandomForestClassifier(n_estimators=100, max_depth=2, random_state=0)
RandForest.fit(train_features, train_labels)
forest_pred = RandForest.predict(test_features)
print(RandForest.score(test_features,test_labels))

In [ ]:
confusion_matrix(test_labels, forest_pred)

### KNN

In [ ]:
correct = []
for i in range(1, 20):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(train_features,train_labels)
    prediction = knn.predict(test_features)
    correct.append(np.mean(prediction == test_labels))

    plt.figure(figsize=(10, 5))
plt.plot(range(1, 20), correct)
plt.title('% Correct vs. K Value')
plt.xlabel('K Value')
plt.ylabel('% Correct')


# K Nearest Neighbor with k=1.
knn = KNeighborsClassifier(n_neighbors = 1)
knn.fit(train_features,train_labels)
print("Accuracy " + str(knn.score(test_features, test_labels)))
y_pred = knn.predict(test_features)
print(confusion_matrix(test_labels, y_pred))


### MLP

In [ ]:
NeuralNetwork = MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5, 2), random_state=1)
NeuralNetwork.fit(train_features, train_labels)
nn_pred = NeuralNetwork.predict(test_features)
print(NeuralNetwork.score(test_features,test_labels))
print(confusion_matrix(test_labels, nn_pred))

### SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
clf = make_pipeline(StandardScaler(), SVC(gamma='auto'))
clf.fit(train_features,train_labels)

In [ ]:
pred = clf.predict(test_features)

In [ ]:
print(confusion_matrix(test_labels, pred))